In [ ]:
import os
import torch
import random
import numpy as np
import xml.etree.ElementTree as ET
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms as T
from torchvision.transforms import functional as F

# === CONFIGURATION ===
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 2  # background + crazing
CONF_THRESHOLD = 0.5
RANDOM_SEED = 42

TRAIN_IMG_ROOT = "./train_images"
TRAIN_ANN_ROOT = "./train_annotations"
VAL_IMG_ROOT = "./test"
MODEL_PATH = "best_model_crazing.pth"
SUBMISSION_PATH = "./submission_crazing.csv"
SAVE_IMG_DIR = "output_crazing"

# === SET SEED FOR REPRODUCIBILITY ===
def set_seed(seed=RANDOM_SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# === DATA AUGMENTATION ===
def get_transform(train=True):
    transforms = []
    if train:
        transforms += [
            T.RandomHorizontalFlip(0.5),
            T.ColorJitter(0.2, 0.2, 0.2, 0.2),
            T.RandomRotation(10)
        ]
    transforms.append(T.ToTensor())
    return T.Compose(transforms)

# === DATASET CLASS ===
class CrazingDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir = img_dir
        self.ann_dir = ann_dir
        self.transforms = transforms
        self.imgs = self._load_valid_images()

    def _load_valid_images(self):
        imgs = []
        for fname in sorted(os.listdir(self.img_dir)):
            if not fname.endswith('.jpg'):
                continue
            ann_path = os.path.join(self.ann_dir, fname.replace('.jpg', '.xml'))
            if not os.path.exists(ann_path):
                continue
            tree = ET.parse(ann_path)
            root = tree.getroot()
            if any(obj.find('name').text == 'crazing' for obj in root.findall('object')):
                imgs.append(fname)
        return imgs

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        ann_path = os.path.join(self.ann_dir, img_name.replace('.jpg', '.xml'))

        image = Image.open(img_path).convert("RGB")
        boxes, labels = [], []

        tree = ET.parse(ann_path)
        for obj in tree.findall('object'):
            if obj.find('name').text != 'crazing':
                continue
            bndbox = obj.find('bndbox')
            bbox = [int(float(bndbox.find(pt).text)) for pt in ['xmin', 'ymin', 'xmax', 'ymax']]
            boxes.append(bbox)
            labels.append(1)

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': torch.tensor([idx])
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

    def __len__(self):
        return len(self.imgs)

def collate_fn(batch):
    return tuple(zip(*batch))

# === MODEL SETUP ===
def get_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model.to(DEVICE)

# === TRAINING ===
def train_model(model, train_loader, epochs=10, save_path=MODEL_PATH):
    model.train()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    best_loss = float('inf')

    for epoch in range(epochs):
        total_loss = 0
        model.train()

        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()

            torch.cuda.empty_cache()

        avg_loss = total_loss / len(train_loader)
        print(f"[INFO] Epoch {epoch+1} - Loss: {avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path)
            print(f"[INFO] Saved best model at epoch {epoch+1} with loss {avg_loss:.4f}")

        scheduler.step()

# === PREDICTION AND EXPORT ===
def predict_and_export(model, val_dir, save_csv_path, save_img_dir=SAVE_IMG_DIR):
    model.eval()
    os.makedirs(save_img_dir, exist_ok=True)
    rows = []

    for img_file in tqdm(sorted(os.listdir(val_dir)), desc="Predicting"):
        if not img_file.lower().endswith(".jpg"):
            continue

        img_path = os.path.join(val_dir, img_file)
        image = Image.open(img_path).convert("RGB")
        img_tensor = F.to_tensor(image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(img_tensor)[0]

        boxes = output['boxes'].cpu().numpy()
        scores = output['scores'].cpu().numpy()
        labels = output['labels'].cpu().numpy()

        cf_list, xmin_list, ymin_list, xmax_list, ymax_list = [], [], [], [], []

        fig, ax = plt.subplots(1)
        ax.imshow(image)

        for box, score, label in zip(boxes, scores, labels):
            if score < CONF_THRESHOLD or label != 1:
                continue
            xmin, ymin, xmax, ymax = map(int, box)
            xmin_list.append(str(xmin))
            ymin_list.append(str(ymin))
            xmax_list.append(str(xmax))
            ymax_list.append(str(ymax))
            cf_list.append(f"{score:.2f}")

            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(xmin, ymin - 5, f"crazing {score:.2f}", color='red', fontsize=8)

        fig.savefig(os.path.join(save_img_dir, img_file), bbox_inches='tight', dpi=150)
        plt.close(fig)

        rows.append({
            "ID": img_file,
            "label": "crazing" if cf_list else "none",
            "cf": " ".join(cf_list),
            "xmin": " ".join(xmin_list),
            "ymin": " ".join(ymin_list),
            "xmax": " ".join(xmax_list),
            "ymax": " ".join(ymax_list),
        })

    df = pd.DataFrame(rows)
    df.fillna("none", inplace=True)
    df.to_csv(save_csv_path, index=False)
    print(f"[INFO] CSV saved to: {save_csv_path}")
    print(f"[INFO] Annotated images saved to: {save_img_dir}")

# === MAIN FUNCTION ===
def main():
    set_seed()
    print("[INFO] Loading dataset...")
    train_dataset = CrazingDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT, transforms=get_transform(train=True))
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

    print("[INFO] Initializing model...")
    model = get_model()

    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print("[INFO] Loaded pre-trained model.")
    else:
        print("[INFO] Training model from scratch...")
    train_model(model, train_loader, epochs=5, save_path=MODEL_PATH)
main()


[INFO] Loading dataset...
[INFO] Initializing model...
[INFO] Loaded pre-trained model.


Epoch 1/5: 100%|██████████| 60/60 [09:12<00:00,  9.21s/it]


[INFO] Epoch 1 - Loss: 0.4364
[INFO] Saved best model at epoch 1 with loss 0.4364


Epoch 2/5: 100%|██████████| 60/60 [09:20<00:00,  9.35s/it]


[INFO] Epoch 2 - Loss: 0.4453


Epoch 3/5: 100%|██████████| 60/60 [09:32<00:00,  9.54s/it]


[INFO] Epoch 3 - Loss: 0.4570


Epoch 4/5:  23%|██▎       | 14/60 [02:40<08:02, 10.50s/it]

In [3]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === CONFIG ===
IMG_DIR = "./train_images"
ANN_DIR = "./train_annotations"
OUT_DIR = "./train_output_vis"

# === LABEL MAP ===
LABEL_MAP = {
    'crazing': 0,
    'inclusion': 1,
    'patches': 2,
    'pitted_surface': 3,
    'rolled-in_scale': 4,
    'scratches': 5
}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

# === CREATE OUTPUT FOLDER ===
os.makedirs(OUT_DIR, exist_ok=True)

# === PROCESS IMAGES ===
image_files = sorted([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])

for img_file in image_files:
    img_path = os.path.join(IMG_DIR, img_file)
    ann_path = os.path.join(ANN_DIR, img_file.replace('.jpg', '.xml'))

    image = Image.open(img_path).convert("RGB")

    fig, ax = plt.subplots(1)
    ax.imshow(image)

    tree = ET.parse(ann_path)
    root = tree.getroot()

    for obj in root.findall('object'):
        label = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(float(bbox.find('xmin').text))
        ymin = int(float(bbox.find('ymin').text))
        xmax = int(float(bbox.find('xmax').text))
        ymax = int(float(bbox.find('ymax').text))

        # Draw bounding box
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                 linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)

        # Label text
        ax.text(xmin, ymin - 5, f"{label}", color='red', fontsize=8, weight='bold')

    # Save the annotated image
    output_path = os.path.join(OUT_DIR, img_file)
    fig.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close(fig)

print(f"[INFO] Saved annotated training images to {OUT_DIR}/")


[INFO] Saved annotated training images to ./train_output_vis/


In [4]:
import os
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# === CONFIG ===
VAL_IMG_ROOT = "./test"
CSV_PATH = "./submission7.csv"
OUT_DIR = "./test_output_vis"

os.makedirs(OUT_DIR, exist_ok=True)

# === LOAD CSV ===
df = pd.read_csv(CSV_PATH)

for idx, row in df.iterrows():
    img_file = row["ID"]
    img_path = os.path.join(VAL_IMG_ROOT, img_file)

    if not os.path.exists(img_path):
        print(f"[WARNING] Image not found: {img_path}")
        continue

    # Load image
    image = Image.open(img_path).convert("RGB")
    fig, ax = plt.subplots(1)
    ax.imshow(image)

    # Parse values
    labels = row["label"].split() if isinstance(row["label"], str) else []
    cfs = row["cf"].split() if isinstance(row["cf"], str) else []
    xmins = row["xmin"].split() if isinstance(row["xmin"], str) else []
    ymins = row["ymin"].split() if isinstance(row["ymin"], str) else []
    xmaxs = row["xmax"].split() if isinstance(row["xmax"], str) else []
    ymaxs = row["ymax"].split() if isinstance(row["ymax"], str) else []

    num_boxes = min(len(cfs), len(xmins), len(ymins), len(xmaxs), len(ymaxs))

    for i in range(num_boxes):
        try:
            xmin = int(xmins[i])
            ymin = int(ymins[i])
            xmax = int(xmaxs[i])
            ymax = int(ymaxs[i])
            cf = float(cfs[i])
            label = row["label"] if isinstance(row["label"], str) else "unknown"

            # Draw box
            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)

            # Label text
            ax.text(xmin, ymin - 5, f"{label} {cf:.2f}", color='red', fontsize=8, weight='bold')
        except Exception as e:
            print(f"[ERROR] Skipping box {i} in {img_file}: {e}")

    # Save annotated image
    out_path = os.path.join(OUT_DIR, img_file)
    fig.savefig(out_path, bbox_inches='tight', dpi=150)
    plt.close(fig)

print(f"[INFO] Annotated test images saved to: {OUT_DIR}/")


[INFO] Annotated test images saved to: ./test_output_vis/


In [6]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 1.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip


In [9]:
import kagglehub
import shutil
import os

# Step 1: Download dataset from KaggleHub
dataset_path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")

print("✅ Download complete.")
print("📁 Original download location:", dataset_path)

# Step 2: Define target directory
target_dir = "/Users/wangdashuai/Desktop/ml"

# Step 3: Copy dataset files to your desired location
shutil.copytree(dataset_path, target_dir, dirs_exist_ok=True)

print(f"✅ Dataset copied to: {target_dir}")


✅ Download complete.
📁 Original download location: /Users/wangdashuai/.cache/kagglehub/datasets/kaustubhdikshit/neu-surface-defect-database/versions/1
✅ Dataset copied to: /Users/wangdashuai/Desktop/ml


In [25]:
import pandas as pd

def filter_low_confidences(input_csv_path, output_csv_path, threshold=0.5):
    df = pd.read_csv(input_csv_path)

    filtered_rows = []
    for _, row in df.iterrows():
        # Convert to string before splitting, replace NaN with empty string
        cf_scores = list(map(float, str(row.get('cf', '')).strip().split()))
        xmin = str(row.get('xmin', '')).strip().split()
        ymin = str(row.get('ymin', '')).strip().split()
        xmax = str(row.get('xmax', '')).strip().split()
        ymax = str(row.get('ymax', '')).strip().split()

        # Filter indices where confidence >= threshold
        keep_indices = [i for i, score in enumerate(cf_scores) if score >= threshold]

        if keep_indices:
            new_row = {
                'ID': row['ID'],
                'label': row['label'],
                'cf': " ".join([f"{cf_scores[i]:.2f}" for i in keep_indices]),
                'xmin': " ".join([xmin[i] for i in keep_indices]),
                'ymin': " ".join([ymin[i] for i in keep_indices]),
                'xmax': " ".join([xmax[i] for i in keep_indices]),
                'ymax': " ".join([ymax[i] for i in keep_indices]),
            }
        else:
            # Optional: keep the row with empty values if all are below threshold
            new_row = {
                'ID': row['ID'],
                'label': "none",
                'cf': "",
                'xmin': "",
                'ymin': "",
                'xmax': "",
                'ymax': "",
            }

        filtered_rows.append(new_row)

    filtered_df = pd.DataFrame(filtered_rows)
    filtered_df.to_csv(output_csv_path, index=False)
    print(f"[INFO] Filtered CSV saved to {output_csv_path}")

# Example usage
filter_low_confidences("submission7.csv", "filtered_submission_new.csv", threshold=0.7)


[INFO] Filtered CSV saved to filtered_submission_new.csv


In [49]:
import pandas as pd

def filter_low_confidences_with_backup(input_csv_path, backup_csv_path, output_csv_path, thresholds):
    df = pd.read_csv(input_csv_path)
    backup_df = pd.read_csv(backup_csv_path)

    filtered_rows = []

    for _, row in df.iterrows():
        label = row.get('label', 'none')
        threshold = thresholds.get(label, 0.5)

        # Convert strings to lists
        cf_scores = list(map(float, str(row.get('cf', '')).strip().split()))
        xmin = str(row.get('xmin', '')).strip().split()
        ymin = str(row.get('ymin', '')).strip().split()
        xmax = str(row.get('xmax', '')).strip().split()
        ymax = str(row.get('ymax', '')).strip().split()

        keep_indices = [i for i, score in enumerate(cf_scores) if score >= threshold]

        if keep_indices:
            new_row = {
                'ID': row['ID'],
                'label': label,
                'cf': " ".join([f"{cf_scores[i]:.2f}" for i in keep_indices]),
                'xmin': " ".join([xmin[i] for i in keep_indices]),
                'ymin': " ".join([ymin[i] for i in keep_indices]),
                'xmax': " ".join([xmax[i] for i in keep_indices]),
                'ymax': " ".join([ymax[i] for i in keep_indices]),
            }
        else:
            new_row = {
                'ID': row['ID'],
                'label': "none",
                'cf': "",
                'xmin': "",
                'ymin': "",
                'xmax': "",
                'ymax': "",
            }

        filtered_rows.append(new_row)

    # Replace 'none' rows with backup
    final_rows = []
    for row in filtered_rows:
        if row['label'] == 'none':
            backup_row = backup_df[backup_df['ID'] == row['ID']]
            if not backup_row.empty:
                replaced_row = backup_row.iloc[0].to_dict()
                final_rows.append(replaced_row)
                print(f"[REPLACED] ID: {row['ID']} replaced with backup row:\n{replaced_row}\n")
            else:
                final_rows.append(row)
        else:
            final_rows.append(row)

    final_df = pd.DataFrame(final_rows)
    final_df.to_csv(output_csv_path, index=False)
    print(f"[INFO] Final CSV saved to {output_csv_path}")


# Example thresholds per label
thresholds_dict = {
    'crazing': 0.5,
    'inclusion': 0.83,
    'patches': 0.63,
    'pitted_surface': 0.6,
    'rolled-in_scale': 0.75,
    'scratches': 0.8,
}

# Run the function
filter_low_confidences_with_backup(
    input_csv_path="submission7.csv",
    backup_csv_path="filtered_submission_0.6.csv",
    output_csv_path="filtered_submission_seperate.csv",
    thresholds=thresholds_dict
)


[REPLACED] ID: rolled-in_scale_9wqogo5ajioz.jpg replaced with backup row:
{'ID': 'rolled-in_scale_9wqogo5ajioz.jpg', 'label': 'rolled-in_scale', 'cf': '0.70', 'xmin': '76', 'ymin': '23', 'xmax': '125', 'ymax': '86'}

[INFO] Final CSV saved to filtered_submission_seperate.csv
